In [ ]:
# %pip install transformers datasets peft accelerate
# %pip install transformers==4.37.2

In [3]:
# %pip install --upgrade transformers peft

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM,TrainingArguments
from transformers import  Trainer, DataCollatorForLanguageModeling, DataCollatorForSeq2Seq
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType
from collections import Counter
import torch


In [ ]:
model_id = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Downloading the models

In [12]:
# Put model in eval mode and move to device (CPU/MPS/GPU)
model.eval()
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

# Your question
input_text = "Instruction: Answer the question\nInput: What is a restaurant"

# Tokenize input and move to device
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_new_tokens=64,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# Decode and print answer
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

a restaurant


In [3]:
# import fitz  # PyMuPDF
# doc = fitz.open('/Users/sheetalsuwalka/Downloads/Documents/Personal Projects/Sheetal Suwalka Resume.pdf')
# print("\n".join([page.get_text() for page in doc]))

In [54]:
dataset = load_dataset("json", data_files="/Users/sheetalsuwalka/Downloads/Documents/Personal Projects/sheetal_data.jsonl")
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 5
    })
})

In [55]:
print(dataset['train']['input'])

["What is Sheetal's Birth place", 'Where did sheetal studied', "Where was sheetal's first Job", 'Where does sheetal works', 'What kind of programming language does sheetal knows']


In [56]:
# Tokenize
# def tokenize_fn(example):
#     prompt = f"Instruction: {example['instruction']}\nInput: {example['input']}\nResponse: {example['output']}"
#     # return prompt
#     return tokenizer(prompt, truncation=True, padding="max_length", max_length=512)



def tokenize_fn(example):
    tokenized_input = tokenizer(example["input"], padding="max_length", truncation=True, max_length=64)
    tokenized_output = tokenizer(example["output"], padding="max_length", truncation=True, max_length=64)
    tokenized_input["labels"] = tokenized_output["input_ids"]
    return tokenized_input

# def tokenize_fn(example):
#     prompt = f"Instruction: {example['instruction']}\nInput: {example['input']}\nResponse: {example['output']}"
#     tokenized = tokenizer(prompt, truncation=True, padding="max_length", max_length=64)
#     tokenized["labels"] = tokenized["input_ids"].copy()
#     return tokenized

# Because models need numerical inputs (input_ids) and corresponding labels (labels) to compute loss and update weights during training. 
# Without this step, the model can't be fine-tuned.


In [57]:
tokenizer.pad_token = tokenizer.eos_token
tokenized = dataset.map(tokenize_fn, remove_columns=["input", "output"])

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [58]:
tokenized['train']

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 5
})

In [59]:
# print(len(tokenized['train'][0]['input_ids']))
for i in range(5):
    input = list(tokenized['train'][i]['input_ids'])
    output = list(tokenized['train'][i]['labels'])
    counter_in = Counter(input)
    counter_out = Counter(output)
    most_common_item_in, count_in = counter_in.most_common(1)[0]
    most_common_item_out, count_out = counter_out.most_common(1)[0]
    print(f"Most frequent item input: {most_common_item_in} Count: {count_in}, output: {most_common_item_out}, Count: {count_out}")
# print(tokenized)

Most frequent item input: 1 Count: 56, output: 1, Count: 37
Most frequent item input: 1 Count: 59, output: 1, Count: 5
Most frequent item input: 1 Count: 56, output: 1, Count: 37
Most frequent item input: 1 Count: 59, output: 1, Count: 48
Most frequent item input: 1 Count: 55, output: 1, Count: 39


In [60]:
from transformers import Seq2SeqTrainer
from transformers import Seq2SeqTrainingArguments

In [61]:
# LoRA config

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"],  # FLAN-T5 uses different naming than GPT-style models
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

# Apply LoRA
model = get_peft_model(model, lora_config)



# Training setup

# args = Seq2SeqTrainingArguments(
#     output_dir="./flan-t5-lora",
#     num_train_epochs=10,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     save_total_limit=1,
#     label_names=["labels"],
#     logging_steps=10,
#     save_strategy="no",
#     optim="adamw_torch",
#     fp16=False,  # MPS does not support fp16
#     predict_with_generate=True,  # required for evaluation/generation
# )


args = TrainingArguments(
    output_dir="./flan-t5-lora",
    per_device_train_batch_size=1,
    num_train_epochs=5,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)


In [15]:
# args = TrainingArguments(
#     output_dir="./flan-t5-lora",
#     num_train_epochs=10,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     save_total_limit=1,
#     label_names=["labels"],
#     logging_steps=10,
#     save_strategy="no",
#     optim="adamw_torch",
#     fp16=False,  # MPS does not support fp16
# )


# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=tokenized["train"],
#     tokenizer=tokenizer,
#     data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
# )

In [62]:
# trainer = Seq2SeqTrainer(
#     model=model,
#     args=args,
#     train_dataset=tokenized["train"],
#     tokenizer=tokenizer,
#     data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
# )

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

# Start training
trainer.train()

/var/folders/y5/sqxvp96j7kq4n2yv8qbqmyc80000gn/T/ipykernel_9405/3194128209.py:9: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/Users/sheetalsuwalka/Downloads/Documents/Personal Projects/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_

Step,Training Loss
1,5.794800
2,5.491900
3,4.027800
4,4.638300
5,5.112100
6,4.977000
7,5.211900
8,4.556100
9,4.996900
10,4.863800


TrainOutput(global_step=25, training_loss=4.897428455352784, metrics={'train_runtime': 4.2644, 'train_samples_per_second': 5.862, 'train_steps_per_second': 5.862, 'total_flos': 584210841600.0, 'train_loss': 4.897428455352784, 'epoch': 5.0})

In [63]:
import torch

In [ ]:
model.eval()
input_text = "Instruction: Answer the question\nInput: What is capital of Rajasthan India "
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    outputs = model.base_model.generate(
        input_ids,
        max_new_tokens=64,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        output_scores=True,
    )
    

print("Decoded output:", tokenizer.decode(outputs.sequences[0], skip_special_tokens=True))



Decoded output: sarajee
